# Select images for analysis

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d00_utils import dirnames as dn
from src.d00_utils import utilities as utils

In [ ]:
input_dirpath = Path(input())

In [ ]:
# Create CSV file with all of the image names
imgnames = [f.name for f in input_dirpath.glob('*.ome.tif')]
imgnames.sort()
df = pd.DataFrame({'image name': imgnames})
df.head()

In [ ]:
namesplits = df['image name'].str.replace('-', '_').str.split('.ome.tif').str[0].str.split('_')
print(namesplits[0])
df['experiment'] = namesplits.str[0]
df['wellID'] = df['experiment'] + '_' + namesplits.str[1] + '-' + namesplits.str[6]
df['scene'] = namesplits.str[5]
df['img idx'] = np.arange(0, len(df))
df.head()

In [ ]:
wellcond_df_path = Path(input())

In [ ]:
wellcond_df = pd.read_csv(wellcond_df_path)
wellcond_df.head()

wellcond_df['wellID'] = wellcond_df['experiment'] + '_' + wellcond_df['wellID']
wellcond_df

In [ ]:
num_rows_premerge = len(df)
#img_list_df = pd.merge(df, wellcond_df, how='left', on=['wellID'], suffixes=['', '_y'], validate='many_to_one')
df = pd.merge(df, wellcond_df, how='left', suffixes=['', '_y'], validate='many_to_one')
num_rows_postmerge = len(df)
assert num_rows_premerge==num_rows_postmerge

df.head()

In [ ]:
tx_sel = ['218 + 281', '218 + 283']
cond = [df['tx'] == tx for tx in tx_sel]
df['selected'] = False
vals = [True] * len(cond)
df['selected'] = np.select(cond, vals)


In [ ]:
tables_dirpath = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname
tables_dirpath.mkdir(exist_ok=True)
tablename = 'selected_imgs.csv'
selected_df = df[df['selected']==True]
utils.safe_save_csv(selected_df, tables_dirpath / tablename)

In [ ]:
df_subset = df[df['selected']==True]

df_subset.groupby(['tx'])['tx'].count()